# 04 — Join features and split full sequences
Reuse/create `samples.pt` below. Missing feature caches are created by stages 02/03. Shared pages stay in the same split. Class weights use original training labels only.


In [1]:
from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
CACHE.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

import random
from collections import defaultdict
import numpy as np
import torch
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'sequences.json', '01_prepare_sequence_cache.ipynb')
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'deberta.pt', '02_cache_deberta_features.ipynb')
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'magi.pt', '03_cache_magi_features.ipynb')
SAMPLES = CACHE / 'samples.pt'
sequences = json.loads((CACHE / 'sequences.json').read_text(encoding='utf-8'))
# Reuse matching features, refresh stale ones.
run_stage('02_cache_deberta_features.ipynb')
run_stage('03_cache_magi_features.ipynb')
semantic = torch.load(CACHE / 'deberta.pt', map_location='cpu', weights_only=True)
visual = torch.load(CACHE / 'magi.pt', map_location='cpu', weights_only=True)
# Fingerprint the inputs/settings so stale feature or sample caches are not silently reused.
signature = fingerprint({'sequences': sequences, 'semantic': semantic['signature'], 'visual': visual['signature'],
                         'seed': CONFIG['seed'], 'val_fraction': CONFIG['val_fraction'], 'beta': CONFIG['beta']})
saved = torch.load(SAMPLES, map_location='cpu', weights_only=True) if SAMPLES.exists() else None
print('Reuse matching samples:', bool(saved and saved['signature'] == signature))


Existing caches: ['deberta.pt', 'magi.pt', 'sequences.json']
Running prerequisite: 02_cache_deberta_features.ipynb
Existing caches: ['deberta.pt', 'magi.pt', 'sequences.json']
Reuse matching cache: True
Semantic sequences: 80
Running prerequisite: 03_cache_magi_features.ipynb
Visual sequences: 80
Reuse matching samples: False


Join all modalities, split intact sequence groups and compute training-only class weights.


In [2]:
# Only run the cache-building branch when the stored fingerprint is absent or stale.
if saved is None or saved['signature'] != signature:
    samples = []
    for seq in sequences:
        sid = seq['sequence_id']
        geometry = []
        # Process boxes in their saved order so every modality refers to the same sequence token.
        for box in seq['boxes']:
            page = seq['pages'][box['page_index']]
            w, h = page['width'], page['height']
            x1, y1, x2, y2 = box['bbox']
            assert w > 0 and h > 0 and x2 > x1 and y2 > y1
            # Normalize center x/y and width/height by the original page dimensions.
            geometry.append([(x1+x2)/(2*w), (y1+y2)/(2*h), (x2-x1)/w, (y2-y1)/h])
        ids = [b['box_id'] for b in seq['boxes']]
        assert len(ids) == len(set(ids))
        # Join feature tensors by box ID; labels=-100 do not remove real context boxes.
        sample = {'sequence_id': sid, 'boxes': seq['boxes'], 'pages': seq['pages'],
            'text': torch.stack([semantic['vectors'][sid][bid] for bid in ids]),
            'visual': torch.stack([visual['vectors'][sid][bid] for bid in ids]),
            'tail': torch.stack([visual['tails'][sid][bid] for bid in ids]),
            'geometry': torch.tensor(geometry, dtype=torch.float),
            'page': torch.tensor([b['page_index'] for b in seq['boxes']], dtype=torch.long),
            'labels': torch.tensor([b['label'] for b in seq['boxes']], dtype=torch.long)}
        for key in ['text', 'visual', 'tail', 'geometry']:
            assert torch.isfinite(sample[key]).all(), (sid, key)
        assert sample['text'].shape == (len(ids), 768)
        assert sample['tail'].shape == (len(ids), 2)
        assert sample['visual'].ndim == 2
        samples.append(sample)

    def counts(indices):
        labels = torch.cat([samples[i]['labels'] for i in indices])
        return torch.bincount(labels[labels != -100], minlength=2)

    # Reuse the exact baseline sequence split rather than selecting another split.
    baseline_split = torch.load(ROOT.parent / 'NarrationImbalanceTraining/cache/samples.pt',
                                map_location='cpu', weights_only=True)
    baseline_samples = baseline_split['samples']
    expected_ids = {s['sequence_id'] for s in baseline_samples}
    assert expected_ids == {s['sequence_id'] for s in samples}
    baseline_pages = {s['sequence_id']: [p['sha256'] for p in s['pages']] for s in baseline_samples}
    assert all([p['sha256'] for p in s['pages']] == baseline_pages[s['sequence_id']] for s in samples)
    val_ids = {baseline_samples[i]['sequence_id'] for i in baseline_split['val_indices']}
    val_idx = [i for i, s in enumerate(samples) if s['sequence_id'] in val_ids]
    train_idx = [i for i in range(len(samples)) if i not in set(val_idx)]
    assert not ({p['sha256'] for i in train_idx for p in samples[i]['pages']} &
                {p['sha256'] for i in val_idx for p in samples[i]['pages']})
    # Count original training labels once, before any sequence oversampling.
    train_counts = counts(train_idx).double()
    assert (train_counts > 0).all() and (counts(val_idx) > 0).all()
    beta = CONFIG['beta']
    assert 0 <= beta < 1
    # Compute effective-number weights stably; beta=0 gives equal class weights.
    weights = torch.ones(2, dtype=torch.double) if beta == 0 else (1-beta) / (-torch.expm1(train_counts * np.log(beta)))
    # Normalize class weights to mean one so their overall scale stays interpretable.
    weights = (weights / weights.mean()).float()
    saved = {'signature': signature, 'samples': samples, 'train_indices': train_idx,
             'val_indices': val_idx, 'train_counts': train_counts.long(), 'class_weights': weights,
             'max_positions': max(len(s['boxes']) for s in samples)}
    torch.save(saved, SAMPLES)
print('Train / validation sequences:', len(saved['train_indices']), len(saved['val_indices']))
print('Original train label counts:', saved['train_counts'].tolist(), '| weights:', saved['class_weights'].tolist())
for split, indices in [('train', saved['train_indices']), ('validation', saved['val_indices'])]:
    labels = torch.cat([saved['samples'][i]['labels'] for i in indices])
    print(split, 'labeled boxes:', torch.bincount(labels[labels != -100], minlength=2).tolist(),
          '| ignored:', int((labels == -100).sum()))


Train / validation sequences: 64 16
Original train label counts: [1142, 50] | weights: [0.13372045755386353, 1.8662796020507812]
train labeled boxes: [1142, 50] | ignored: 572
validation labeled boxes: [285, 13] | ignored: 121
